# LOADING DATA

In [1]:
import pandas as pd
import pickle
from pathlib import Path

# Get the Code directory (project root)
current_dir = Path.cwd()  # from_scratch directory
code_dir = current_dir.parent.parent.parent.parent  # Go up to Code directory
print(f"Code directory: {code_dir}")

# Define all data paths directly
PATHS = {
    # Training features
    'X_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote.parquet',
    'X_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_tomek.parquet',
    'X_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote_tomek.parquet',
    
    # Training targets
    'y_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote' / 'y_smote.pkl',
    'y_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'tomek' / 'y_tomek.pkl',
    'y_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote_tomek' / 'y_smote_tomek.pkl',
    
    # Validation and test features
    'X_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'X_val.parquet',
    'X_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'X_test.parquet',
    
    # Validation and test targets
    'y_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'y_val.pkl',
    'y_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'y_test.pkl',
}

# Print all paths for verification
print("\nData paths:")
for key, path in PATHS.items():
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {key}: {path}")

# Load all data
def load_all_data():
    """Load all data files"""
    data = {}
    
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)
    
    # Load parquet files
    parquet_keys = ['X_train_smote', 'X_train_tomek', 'X_train_smote_tomek', 'X_val', 'X_test']
    for key in parquet_keys:
        path = PATHS[key]
        if path.exists():
            try:
                data[key] = pd.read_parquet(path)
                print(f"✓ Loaded {key}: {data[key].shape}")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    # Load pickle files
    pickle_keys = ['y_train_smote', 'y_train_tomek', 'y_train_smote_tomek', 'y_val', 'y_test']
    for key in pickle_keys:
        path = PATHS[key]
        if path.exists():
            try:
                with open(path, 'rb') as f:
                    data[key] = pickle.load(f)
                print(f"✓ Loaded {key}: {len(data[key])} samples")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    return data

# Load the data
data = load_all_data()

if data:
    print(f"\n" + "="*50)
    print(f"Successfully loaded {len(data)} datasets")
    print("="*50)
    for key, value in data.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: {value.shape}")
        else:
            print(f"  {key}: {len(value)} samples")
else:
    print("\nNo data was loaded")

Code directory: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code

Data paths:
  ✓ X_train_smote: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote.parquet
  ✓ X_train_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_tomek.parquet
  ✓ X_train_smote_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote_tomek.parquet
  ✓ y_train_smote: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\smote\y_smote.pkl
  ✓ y_train_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\tomek\y_tomek.pkl
  ✓ y_train_smote_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessi

# RF

In [2]:
import numpy as np
import pandas as pd
import time
# Import Scikit-learn libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Re-define the accuracy function for consistency (though we'll use accuracy_score)
def calculate_accuracy(y_true, y_pred):
    """Calculate the accuracy score using sklearn."""
    # Ensure inputs are Series/arrays for compatibility
    if isinstance(y_true, pd.Series):
        y_true = y_true.values
    return accuracy_score(y_true, y_pred)

# --- Training and Evaluation Function ---

def train_and_evaluate_sklearn_rf(data, train_key_X, train_key_y, val_key_X, val_key_y, model_params):
    """Utility function to train and evaluate the Scikit-learn RF on a specific dataset."""
    dataset_name = train_key_X.upper().replace('X_TRAIN_', '')
    print(f"\n--- Training Scikit-learn Random Forest on **{dataset_name}** Dataset ---")
    
    # Prepare data (Scikit-learn handles DataFrame input directly)
    X_train = data[train_key_X]
    y_train = data[train_key_y]
    X_val = data[val_key_X]
    y_val = data[val_key_y]
    
    # Initialize and train the model
    rf = RandomForestClassifier(**model_params)
    start_time = time.time()
    
    # Fit the model
    rf.fit(X_train, y_train)
    
    end_time = time.time()
    
    print(f"\nTraining completed in **{end_time - start_time:.2f} seconds**.")
    
    # Make predictions
    y_train_pred = rf.predict(X_train)
    y_val_pred = rf.predict(X_val)
    
    # Evaluate
    train_acc = calculate_accuracy(y_train, y_train_pred)
    val_acc = calculate_accuracy(y_val, y_val_pred)
    
    print(f"**Training Accuracy:** {train_acc:.4f}")
    print(f"**Validation Accuracy:** {val_acc:.4f}")

    # Optional: Print detailed report for validation set
    print("\nClassification Report (Validation):")
    print(classification_report(y_val, y_val_pred))
    
    return rf, train_acc, val_acc

# Define model parameters (similar to the scratch version, but using sklearn names)
SKLEARN_RF_PARAMS = {
    'n_estimators': 50,          # Number of trees
    'max_depth': 10,             # Max depth of each tree
    'max_features': 0.5,         # Number of features to consider for best split (similar to n_features_ratio)
    'bootstrap': True,           # Enable bootstrapping (default is True)
    'random_state': 42,
    'n_jobs': -1                 # Use all processor cores for faster training
}

# List of datasets to process
datasets = [
    ('X_train_smote', 'y_train_smote'),
    ('X_train_tomek', 'y_train_tomek'),
    ('X_train_smote_tomek', 'y_train_smote_tomek'),
]

results = {}
val_features_key = 'X_val'
val_target_key = 'y_val'

# Run the training loop for all sampled datasets
for X_key, y_key in datasets:
    model, train_acc, val_acc = train_and_evaluate_sklearn_rf(
        data, 
        X_key, 
        y_key, 
        val_features_key, 
        val_target_key, 
        SKLEARN_RF_PARAMS
    )
    # Store results
    dataset_name = X_key.replace('X_train_', '')
    results[dataset_name] = {
        'model': model,
        'train_accuracy': train_acc,
        'val_accuracy': val_acc
    }

# --- Final Summary ---
print("\n" + "="*70)
print("FINAL SCIKIT-LEARN RANDOM FOREST TRAINING RESULTS")
print("="*70)
for name, res in results.items():
    print(f"🌳 **{name.upper()}**:")
    print(f"  - Training Accuracy: {res['train_accuracy']:.4f}")
    print(f"  - Validation Accuracy: {res['val_accuracy']:.4f}")
    print("-" * 25)


--- Training Scikit-learn Random Forest on **SMOTE** Dataset ---

Training completed in **4.61 seconds**.
**Training Accuracy:** 0.9599
**Validation Accuracy:** 0.9008

Classification Report (Validation):
              precision    recall  f1-score   support

           0       0.98      0.91      0.95      3276
           1       0.33      0.68      0.44       201

    accuracy                           0.90      3477
   macro avg       0.65      0.80      0.69      3477
weighted avg       0.94      0.90      0.92      3477


--- Training Scikit-learn Random Forest on **TOMEK** Dataset ---

Training completed in **1.29 seconds**.
**Training Accuracy:** 0.9745
**Validation Accuracy:** 0.9471

Classification Report (Validation):
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      3276
           1       0.57      0.36      0.44       201

    accuracy                           0.95      3477
   macro avg       0.76      0.67      0.71

# 🔬 How Bayesian Optimization WorksBayesian Optimization
 (BO) works by treating the validation accuracy as a function $f(\mathbf{x})$ where $\mathbf{x}$ are the hyperparameters. BO aims to find the $\mathbf{x}$ that maximizes $f(\mathbf{x})$ using minimal evaluations.Gaussian Process (GP): BO uses a GP to model the unknown function $f$. After each trial, the GP is updated to reflect the new score, making the model more confident in predicting the performance of untried hyperparameter combinations.Acquisition Function: This function (e.g., Expected Improvement, Upper Confidence Bound) uses the GP model to determine which set of hyperparameters to test next. It balances exploration (testing new areas) and exploitation (refining known good areas). This intelligent selection is what makes BO faster than traditional methods.The output will provide the final, tuned parameters that gave the highest cross-validation score, which you then use to train your final model.

In [3]:
!pip install bayesian-optimization

In [4]:
import numpy as np
import pandas as pd
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score

# --- Define the Objective Function for BO ---

def rf_cv_score(n_estimators, max_depth, max_features):
    """
    Objective function for Bayesian Optimization.
    It takes hyperparameter suggestions and returns the cross-validation score.
    
    Parameters from BO are typically floats, so we cast them to appropriate types.
    """
    
    # 1. Cast parameters to correct types
    n_estimators = int(round(n_estimators))
    max_depth = int(round(max_depth))
    
    # max_features should be between 0.1 and 1.0 (or 'sqrt', 'log2', etc.)
    # We will use a float ratio for simplicity in BO
    max_features = max(0.01, min(1.0, max_features)) # Ensure max_features is within [0.01, 1.0]

    # 2. Define the Random Forest model
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_features=max_features,
        random_state=42,
        n_jobs=-1,  # Use all cores for speed
        criterion='gini' # Use Gini impurity
    )

    # 3. Perform Cross-Validation on the training data
    # We use a smaller 3-fold CV for speed, but 5-fold is often standard.
    # Note: Using the combined X_train/y_train data for CV here.
    try:
        scores = cross_val_score(model, X_train_opt, y_train_opt, cv=3, scoring='accuracy', error_score='raise')
        return scores.mean()
    except Exception as e:
        # In case of training error (e.g., max_features too low)
        print(f"Error during CV: {e}")
        return 0.0

# --- Load and Prepare Data for Optimization ---

# Select the SMOTE-Tomek dataset for optimization
X_train_opt = data['X_train_smote_tomek']
y_train_opt = data['y_train_smote_tomek']

# 1. Define the search space (Parameter Bounds)
# BO explores this space intelligently
pbounds = {
    # Number of trees (must be an integer, BO suggests floats)
    'n_estimators': (50, 200),
    
    # Maximum depth of the tree (must be an integer)
    'max_depth': (5, 25),
    
    # Fraction of features to consider for best split (float)
    'max_features': (0.1, 0.9), 
}

print(f"Starting Bayesian Optimization on SMOTE-Tomek dataset...")
print(f"Search space: {pbounds}")
print("-" * 50)

# 2. Initialize the Bayesian Optimizer
optimizer = BayesianOptimization(
    f=rf_cv_score,        # The function to maximize
    pbounds=pbounds,      # The parameter space
    random_state=42,      
    verbose=2             # 1: prints steps, 2: prints steps and detailed results
)

# 3. Run the Optimization
# 'init_points': number of random exploration steps (initial samples)
# 'n_iter': number of optimization steps using the acquisition function (exploitation)
start_time = time.time()
optimizer.maximize(
    init_points=5,
    n_iter=15
)
end_time = time.time()

print("\n" + "="*50)
print(f"BAYESIAN OPTIMIZATION COMPLETE in {end_time - start_time:.2f} seconds.")
print("="*50)

# --- Extract Best Parameters and Final Evaluation ---

best_params_raw = optimizer.max['params']
best_score = optimizer.max['target']

# Clean up and finalize the best parameters
best_params = {
    'n_estimators': int(round(best_params_raw['n_estimators'])),
    'max_depth': int(round(best_params_raw['max_depth'])),
    'max_features': best_params_raw['max_features'], # Keep as float
    'random_state': 42,
    'n_jobs': -1
}

print(f"Optimal CV Accuracy Found: **{best_score:.4f}**")
print("Optimal Hyperparameters:")
for key, value in best_params.items():
    print(f"  - {key}: {value}")
    
# --- Final Model Training with Optimal Parameters ---

print("\n" + "-"*50)
print("FINAL TRAINING on SMOTE-TOMEK with Optimized Parameters")
print("-" * 50)

# 1. Train the final model
final_rf_model = RandomForestClassifier(**best_params)
final_rf_model.fit(data['X_train_smote_tomek'], data['y_train_smote_tomek'])

# 2. Evaluate on Validation Set
X_val = data['X_val']
y_val = data['y_val']

y_val_pred = final_rf_model.predict(X_val)
final_val_acc = accuracy_score(y_val, y_val_pred)

print(f"✅ Final Optimized Validation Accuracy: **{final_val_acc:.4f}**")

Starting Bayesian Optimization on SMOTE-Tomek dataset...
Search space: {'n_estimators': (50, 200), 'max_depth': (5, 25), 'max_features': (0.1, 0.9)}
--------------------------------------------------
|   iter    |  target   | n_esti... | max_depth | max_fe... |
-------------------------------------------------------------
| 1         | 0.9691404 | 106.18101 | 24.014286 | 0.6855951 |
| 2         | 0.9236437 | 139.79877 | 8.1203728 | 0.2247956 |
| 3         | 0.9678188 | 58.712541 | 22.323522 | 0.5808920 |
| 4         | 0.8682350 | 156.21088 | 5.4116898 | 0.8759278 |
| 5         | 0.9353069 | 174.86639 | 9.2467822 | 0.2454599 |
| 6         | 0.9695369 | 108.51175 | 24.209122 | 0.6364646 |
| 7         | 0.9658364 | 78.922239 | 15.928354 | 0.2576000 |
| 8         | 0.8575628 | 106.75422 | 5.0       | 0.1       |
| 9         | 0.9355382 | 65.267724 | 9.1002184 | 0.9       |
| 10        | 0.9702638 | 71.170994 | 25.0      | 0.1       |
| 11        | 0.9676866 | 88.180260 | 25.0      | 0.9   